##### 1 SVM is a margin-based classifier. It finds a decision boundary (hyperplane) that maximizes the margin between two classes. The goal is to find the hyperplane that maximizes the distance (or margin) between the hyperplane and the closest data points of each class. When data is not linearly separable, SVM can use a kernel trick to implicitly map input features into a higher-dimensional space where a linear separator can be found.

##### 2 The training methodology is as follows: first we load all neccesary packages to complete out training. Then we use a label encoder, because our dataset has [-1,1] label values (this sets the values to 0 and 1 respectively). Then we load our training dataset, and use the label encoder to transform the values. Next, we divide the data into train and test on a 60/40 split. Then, we establish our hyper-parameters for the model (we start out with the default values that you can find on any SVM documentation website), using cross validation, we can loop through multiple values to find the best combination of hyperparameters. We then fit the SVM model on the training data and hyper-parameters we set, based on the validation set performance. Finally, we make a prediction on the testing data and evaluate it.

##### 3 The hyperparameters are as follows: kernel, C, and gamma. kernel specifies the kernel type to be used in the algorithm and its default value is kernel = 'rbf'. C is the Regularization parameter, the strength of the regularization is inversely proportional to C and must be strictly positive, and the default value is C = 1.0. gamma is Kernel coefficient for 'rbf', and its default value is gamma = 'scale'. The final values are as follows: kernel = 'rbf', C = 10, gamma = 0.01.

##### 4 The train accuracy I got from XGBOOST was 85.38%, hold-out accuracy was 84.87%, and test accuracy was 85.07%

##### 5 Final Accuracy: 85.07%

In [ ]:
# Problem 3 NN -- CMPSC 448 HW 4
# Aidan Vesci AJV5723

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_svmlight_file
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

le = LabelEncoder()

# load data in LibSVM sparse data forma
X, y = load_svmlight_file("/content/a9a.txt")  # Training set
# X_test, y_test = load_svmlight_file("a9a.t") # Test set

le.fit(y)
y_transformed = le.transform(y)

# split data into train and test sets
seed = 6
test_size = 0.4
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=test_size, random_state=seed)


# kernel = "rbf"
# C = 1.0
# gamma = "scale"


# fit model on training data
model = KNeighborsClassifier()

model.fit(X_train, y_train)

# make predictions for test data
y_pred = model.predict(X_test)
predictions = [round(value) for value in y_pred]
# evaluate predictions
accuracy = accuracy_score(y_test, predictions)
print("BaseAccuracy from Nearest Neighbors: %.2f%%" % (accuracy * 100.0))


BaseAccuracy from Nearest Neighbors: 82.69%


In [ ]:
# Problem 3 SVM -- CMPSC 448 HW 4
# Aidan Vesci AJV5723

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_svmlight_file
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.svm import SVC

le = LabelEncoder()

# load data in LibSVM sparse data forma
X, y = load_svmlight_file("/content/a9a.txt")  # Training set
# X_test, y_test = load_svmlight_file("a9a.t") # Test set

le.fit(y)
y_transformed = le.transform(y)

# split data into train and test sets
seed = 6
test_size = 0.4
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=test_size, random_state=seed)


kernel = "rbf"
C = 8.0
gamma = 0.05


# fit model on training data
model = SVC(kernel = kernel, C = C, gamma = gamma)

model.fit(X_train, y_train)

#### TRAIN ERROR
y_pred = model.predict(X_train)
predictions = [round(value) for value in y_pred]

# evaluate predictions
accuracy = accuracy_score(y_train, predictions)
print("Train Accuracy: %.2f%%" % (accuracy * 100.0))


#### VALIDATION ERROR
y_pred = model.predict(X_test)
predictions = [round(value) for value in y_pred]

# evaluate predictions
accuracy = accuracy_score(y_test, predictions)
print("Validation Accuracy: %.2f%%" % (accuracy * 100.0))


#### TEST ERROR
# 1) comment in code below


# X, y = load_svmlight_file("/content/a9a.t") # test set
# y = le.transform(y)
# preds = model.predict(X)
# accuracy = accuracy_score(y, preds)
# print("Test Accuracy: %.2f%%" % (accuracy * 100.0))



Train Accuracy: 87.61%
Validation Accuracy: 84.70%


In [ ]:
# Problem 3 SVM with Cross-Validation -- CMPSC 448 HW 4
# Aidan Vesci AJV5723

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_svmlight_file
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
import numpy as np

# Load LibSVM training data
X, y = load_svmlight_file("/content/a9a.txt")
le = LabelEncoder()
y_transformed = le.fit_transform(y)

# Hold-out split for validation
seed = 6
test_size = 0.4
X_train, X_val, y_train, y_val = train_test_split(X, y_transformed, test_size=test_size, random_state=seed)

# Define SVM model and hyperparameter grid
model = SVC(kernel="rbf")

param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [0.01, 0.05, 0.1]
}

# Grid search with 5-fold cross-validation on training set
grid = GridSearchCV(estimator=model,
                    param_grid=param_grid,
                    cv=5,
                    scoring='accuracy',
                    verbose=1,
                    n_jobs=-1)

grid.fit(X_train, y_train)

# Best model + hyperparameters
best_model = grid.best_estimator_
print("\nBest Hyperparameters:", grid.best_params_)

#### TRAIN ACCURACY
train_preds = best_model.predict(X_train)
train_acc = accuracy_score(y_train, train_preds)
print("Train Accuracy: {:.2f}%".format(train_acc * 100))

#### VALIDATION ACCURACY
val_preds = best_model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print("Validation Accuracy: {:.2f}%".format(val_acc * 100))

#### TEST ACCURACY

X_test_final, y_test_final = load_svmlight_file("/content/a9a.t")
y_test_final = le.transform(y_test_final)
test_preds = best_model.predict(X_test_final)
test_acc = accuracy_score(y_test_final, test_preds)
print("Test Accuracy: {:.2f}%".format(test_acc * 100))

Fitting 5 folds for each of 9 candidates, totalling 45 fits

Best Hyperparameters: {'C': 10, 'gamma': 0.01}
Train Accuracy: 85.38%
Validation Accuracy: 84.87%
Test Accuracy: 85.07%
